## Clean and save spontaneous speeds

Cleans spontaneous data and saves speed arrays for all fish.

Import libraries:

In [2]:
# --- Imports and path setup (paths from code/paths.py) ---
import numpy as np
import matplotlib.pyplot as plt
def inline():
    %matplotlib inline
    plt.close('all')
def widget():
    %matplotlib widget
    plt.close('all')
import sys
from pathlib import Path
_code = Path.cwd() if (Path.cwd() / "paths.py").exists() else Path.cwd().parent
if str(_code.resolve()) not in [str(Path(p).resolve()) for p in sys.path]:
    sys.path.insert(0, str(_code.resolve()))
from paths import TRACKING_DIR, SPEED_SEGMENTED_DIR
path = str(TRACKING_DIR)
path_save = str(SPEED_SEGMENTED_DIR)

%load_ext autoreload
%autoreload 2
from functions.visual_flow_behavior_plotting import *
from functions.visual_flow_behavior_analysis import *
from functions.brain_analysis import *
from functions.utilities import *
from functions.feedback_models import *
from functions.freely_swimming import *

import os
import h5py
from ipywidgets import interact, IntSlider
from scipy import fft
from scipy.ndimage import gaussian_filter
from skimage import io
import pickle
from sklearn.preprocessing import scale
from mpl_toolkits.axes_grid1.axes_divider import make_axes_locatable
from scipy.stats import iqr
import pandas as pd

In [6]:
# path, path_save set in first cell via paths.TRACKING_DIR and paths.SPEED_SEGMENTED_DIR

#### Import data:

In [5]:
speed = load_h5(path, 'SPEED')[0]

📥 Loaded SPEED ← ../../data/tracking/SPEED.h5


In [ ]:
theta = pd.read_hdf(os.path.join(path, f"THETA_obs.h5"), key='df')

In [ ]:
pose = pd.read_hdf(os.path.join(path, f"POSE.h5"), key='df')

In [ ]:
fish_ids=theta['ID'].unique()
fish_ids

In [ ]:
len(fish_ids)

In [14]:
fish_ids_young=['13_06May24_1','13_30Oct24_3','13_30Oct24_4','14_27Mar24_1','14_31Oct24_1']

In [15]:
fish_ids_intermediate=['17_21May24_2','18_22May24_1','18_22May24_2','19_23May24_5','20_05Jun24_1','20_05Jun24_2','20_12Jun24_1','21_06Dec24_1','21_06Dec24_2','21_06Jun24_1','25_10Jun24_1']

In [16]:
fish_ids_old=['26_02May24_2','27_12Jun24_1','28_04May24_1','30_06May24_2','34_11Dec24_1','34_11Dec24_3']

In [17]:
fish_ids=fish_ids_young+fish_ids_intermediate+fish_ids_old

In [18]:
len(fish_ids)

22

In [19]:
speed_clean = pd.DataFrame()

for fish_ID in fish_ids:
    speed_fish,theta_fish,pose_fish=subset_by_fish(fish_ID,DF1=speed,DF2=theta,DF3=pose)
    L=theta_fish['L_av'].values[0]
    thetas_fish = pose_fish.filter(like='theta').values

    ttail=theta_fish['t'].values
    angletail=theta_fish['theta'].values
    
    tfish=speed_fish['t'].values
    vfish=speed_fish['speed_gauss'].values

    tail_points=compute_tail_points_from_pose_angles(thetas_fish,L)

    mean_thrust=calculate_mean_thrust_from_pose(tail_points,L)

    t_c,vfish_c,vtail_c,dangletail_c,angletail_c,freqtail_c,mean_thrust_c=clean_data_and_interpolate_with_thrust(tfish,vfish,ttail,angletail,mean_thrust,L)
    
    speed_clean_fish = pd.DataFrame({
                            'ID': fish_ID,
                            't_c': t_c,
                            'vfish_c': vfish_c,
                            'vtail_c': vtail_c,
                            'dangletail_c': dangletail_c,
                            'angletail_c': angletail_c,
                            'freqtail_c': freqtail_c,
                            'mean_thrust_c': mean_thrust_c,
                            })
    
    speed_clean=pd.concat([speed_clean, speed_clean_fish], ignore_index=True)

Subsetting data of the fish 13_06May24_1


100%|████████████████████████████████| 143367/143367 [00:07<00:00, 17926.55it/s]


Subsetting data of the fish 13_30Oct24_3


100%|████████████████████████████████| 234565/234565 [00:12<00:00, 18332.41it/s]


Subsetting data of the fish 13_30Oct24_4


...

100%|████████████████████████████████| 276078/276078 [00:17<00:00, 15800.02it/s]


Subsetting data of the fish 34_11Dec24_1


100%|████████████████████████████████| 153973/153973 [00:10<00:00, 15055.33it/s]


Subsetting data of the fish 34_11Dec24_3


100%|██████████████████████████████████| 68198/68198 [00:03<00:00, 17542.14it/s]


Save:

Load:

In [22]:
speed_clean

,ID,t_c,vfish_c,vtail_c,dangletail_c,angletail_c,freqtail_c,mean_thrust_c
0,13_06May24_1,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
1,13_06May24_1,0.003333,NaN,NaN,NaN,NaN,NaN,NaN
2,13_06May24_1,0.006667,NaN,NaN,NaN,NaN,NaN,NaN
3,13_06May24_1,0.010000,NaN,NaN,NaN,NaN,NaN,NaN
4,13_06May24_1,0.013333,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
21727813,34_11Dec24_3,320.736667,NaN,NaN,NaN,NaN,NaN,NaN
21727814,34_11Dec24_3,320.740000,NaN,NaN,NaN,NaN,NaN,NaN
21727815,34_11Dec24_3,320.743333,NaN,NaN,NaN,NaN,NaN,NaN
21727816,34_11Dec24_3,320.746667,NaN,NaN,NaN,NaN,NaN,NaN
